# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI, base_url

In [16]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
OPENAI_MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [59]:
# Initialize and constants with Claude through OpenAI sdk
import anthropic

load_dotenv(override=True)
api_key = os.getenv('ANTHROPIC_API_KEY')

claude = OpenAI(base_url="https://api.anthropic.com/v1/", api_key=api_key)
CLAUDE_MODEL = "claude-haiku-4-5" # or claude-sonnet-4-5
CLAUDE_MODEL = "claude-sonnet-4-5"

In [50]:
# Initialize and constants with ANTHROPIC native sdk
import anthropic

load_dotenv(override=True)
api_key = os.getenv('ANTHROPIC_API_KEY')

claude = anthropic.Anthropic(api_key=api_key)
CLAUDE_MODEL = "claude-haiku-4-5" # or claude-sonnet-4-5
CLAUDE_MODEL = "claude-sonnet-4-5"

In [51]:
# Make a chat completion request
models = claude.models.list()
print("Available Claude Models:")
print("-" * 50)
for model in models.data:
    print(f"Model ID: {model.id}")
    print(f"Display Name: {model.display_name}")
    print(f"Created: {model.created_at}")
    print("-" * 50)

Available Claude Models:
--------------------------------------------------
Model ID: claude-opus-4-6
Display Name: Claude Opus 4.6
Created: 2026-02-04 00:00:00+00:00
--------------------------------------------------
Model ID: claude-opus-4-5-20251101
Display Name: Claude Opus 4.5
Created: 2025-11-24 00:00:00+00:00
--------------------------------------------------
Model ID: claude-haiku-4-5-20251001
Display Name: Claude Haiku 4.5
Created: 2025-10-15 00:00:00+00:00
--------------------------------------------------
Model ID: claude-sonnet-4-5-20250929
Display Name: Claude Sonnet 4.5
Created: 2025-09-29 00:00:00+00:00
--------------------------------------------------
Model ID: claude-opus-4-1-20250805
Display Name: Claude Opus 4.1
Created: 2025-08-05 00:00:00+00:00
--------------------------------------------------
Model ID: claude-opus-4-20250514
Display Name: Claude Opus 4
Created: 2025-05-22 00:00:00+00:00
--------------------------------------------------
Model ID: claude-sonnet-4

In [6]:
links = fetch_website_links("https://ase.ch")
links

['#main',
 'https://ase.ch/en/',
 'https://ase.ch/en/expertises/',
 'https://ase.ch/en/expertises/',
 'https://ase.ch/en/expertises/technology/',
 'https://ase.ch/en/expertises/products/',
 'https://ase.ch/en/expertises/research-development/',
 'https://ase.ch/en/expertises/partners/',
 '#',
 'https://ase.ch/en/industries/mobility/',
 'https://ase.ch/en/industries/mobility/stations/',
 'https://ase.ch/en/industries/mobility/public-transport/',
 'https://ase.ch/en/industries/mobility/traffic/',
 'https://ase.ch/en/industries/mobility/asset-management/',
 'https://ase.ch/en/industries/mobility/ships/',
 'https://ase.ch/en/industries/mobility/airports/',
 'https://ase.ch/en/retail-services/',
 'https://ase.ch/en/retail-services/shopping-center/',
 'https://ase.ch/en/retail-services/retail-chains/',
 'https://ase.ch/en/building-campus/',
 'https://ase.ch/en/building-campus/building/',
 'https://ase.ch/en/building-campus/campus/',
 'https://ase.ch/en/attraction-destination/',
 'https://ase.

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [54]:
# System prompt
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
Output only JSON in the following format:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [14]:
## user prompt
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [9]:
print(get_links_user_prompt("https://ase.ch"))


Here is the list of links on the website https://ase.ch -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#main
https://ase.ch/en/
https://ase.ch/en/expertises/
https://ase.ch/en/expertises/
https://ase.ch/en/expertises/technology/
https://ase.ch/en/expertises/products/
https://ase.ch/en/expertises/research-development/
https://ase.ch/en/expertises/partners/
#
https://ase.ch/en/industries/mobility/
https://ase.ch/en/industries/mobility/stations/
https://ase.ch/en/industries/mobility/public-transport/
https://ase.ch/en/industries/mobility/traffic/
https://ase.ch/en/industries/mobility/asset-management/
https://ase.ch/en/industries/mobility/ships/
https://ase.ch/en/industries/mobility/airports/
https://ase.ch/en/retail-services/
https://ase.ch/en/retail-services/shopping-center/
https://ase.ch/en/retail-serv

In [48]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {OPENAI_MODEL}")
    response = openai.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"} # extra parameter to force output format
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links


In [60]:
# CLAUDE VERSION
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {CLAUDE_MODEL}")
    response = claude.chat.completions.create(
        model=CLAUDE_MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "link_selection",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {
                        "links": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "type": {
                                        "type": "string",
                                        "description": "Type of page (e.g., 'about page', 'careers page')"
                                    },
                                    "url": {
                                        "type": "string",
                                        "description": "Full URL of the page"
                                    }
                                },
                                "required": ["type", "url"],
                                "additionalProperties": False
                            }
                        }
                    },
                    "required": ["links"],
                    "additionalProperties": False
                }
            }
        }
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [57]:
# CLAUDE NATIVE SDK VERSION - DOESNT RUN
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {CLAUDE_MODEL} through native SDK")
    response = claude.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=1024,
        system=link_system_prompt,
        messages=[
            {"role": "user", "content": get_links_user_prompt(url)}
        ]
    )
    result = response.content[0].text
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [61]:
select_relevant_links("https://ase.ch")

Selecting relevant links for https://ase.ch by calling claude-sonnet-4-5
Found 13 relevant links


{'links': [{'type': 'about page', 'url': 'https://ase.ch/en/about-ase/'},
  {'type': 'team page', 'url': 'https://ase.ch/en/about-ase/team/'},
  {'type': 'careers page', 'url': 'https://ase.ch/en/about-ase/jobs/'},
  {'type': 'references page',
   'url': 'https://ase.ch/en/about-ase/references/'},
  {'type': 'certifications page',
   'url': 'https://ase.ch/en/about-ase/certifications/'},
  {'type': 'news page', 'url': 'https://ase.ch/en/news/'},
  {'type': 'contact page', 'url': 'https://ase.ch/en/contact/'},
  {'type': 'expertises page', 'url': 'https://ase.ch/en/expertises/'},
  {'type': 'technology page',
   'url': 'https://ase.ch/en/expertises/technology/'},
  {'type': 'products page', 'url': 'https://ase.ch/en/expertises/products/'},
  {'type': 'research & development page',
   'url': 'https://ase.ch/en/expertises/research-development/'},
  {'type': 'partners page', 'url': 'https://ase.ch/en/expertises/partners/'},
  {'type': 'industries page',
   'url': 'https://ase.ch/en/industr

In [47]:
select_relevant_links("https://huggingface.co")

{'links': [{'type': 'company page', 'url': 'https://huggingface.co/'},
  {'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'Chinese community',
   'url': 'https://www.zhihu.com/org/huggingface'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [39]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [62]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling claude-sonnet-4-5
Found 7 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
zai-org/GLM-5
Updated
about 18 hours ago
•
66.8k
•
1.01k
MiniMaxAI/MiniMax-M2.5
Updated
about 5 hours ago
•
6.09k
•
437
openbmb/MiniCPM-SALA
Updated
3 days ago
•
2.57k
•
412
moonshotai/Kimi-K2.5
Updated
9 days ago
•
726k
•
2.15k
Qwen/Qwen3-Coder-Next
Updated
11 days ago
•
249k
•
842
Browse 2M+ models
Spaces
Running
on
A100
Featured
362
ACE-Step v1.5
🎵
362
Music Generation Foundation Model v1.5
Running
629
Demo Playground
⚡
629
Free platform to access multiple AI models
Running
Featured
4.64k
Wan2.2 Animate
👁
4.64k
Wan2.2 Animat

In [63]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [64]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [65]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling claude-sonnet-4-5
Found 6 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nzai-org/GLM-5\nUpdated\nabout 18 hours ago\n•\n66.8k\n•\n1.01k\nMiniMaxAI/MiniMax-M2.5\nUpdated\nabout 5 hours ago\n•\n6.09k\n•\n437\nopenbmb/MiniCPM-SALA\nUpdated\n3 days ago\n•\n2.57k\n•\n412\nmoonshotai/Kimi-K2.5\nUpdated\n9 days ago\n•\n726k\n•\n2.15k\nQwen/Qwen3-Coder-Next\nUpdated\n11 days ago\n•\n249k\n•\n842\nBrowse 2M+ models\nSpaces\nRunning\non\nA100\nFeatured\n362\nACE-Step v1.5

In [69]:
def create_brochure(company_name, url):
    print(f"Creating brochure for {company_name} at {url} by calling gpt-4.1-mini")
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [68]:
create_brochure("ASE AG", "https://ase.ch")

Selecting relevant links for https://ase.ch by calling claude-sonnet-4-5
Found 13 relevant links


# ASE AG Brochure

---

## About ASE AG  
ASE AG (Analysis Simulation Engineering) is a Swiss-based expert company, founded in 1996 and headquartered in Zurich, specializing in AI and IoT-driven intelligent solutions designed to optimize **mobility, public transport, retail, infrastructure**, and more. With a team of approximately 30 skilled professionals, ASE delivers data-driven solutions that maximize space utilization, enhance safety, and improve customer experience at transport hubs, shopping centers, buildings, campuses, and tourist attractions.  

### Our Mission  
We believe in unlocking the full potential of existing infrastructure and public spaces to increase capacity, reduce operational costs, and foster safer, more satisfying environments without the need for costly new investments.

---

## Expertise & Technologies  

ASE’s core competencies focus on leveraging **real-time data, AI, and expert consulting** to enable intelligent decisions in these fields:  

- **People Counting & Crowd Monitoring**  
- **Vehicle Recognition**  
- **AI Detection & Simulations**  
- **Live & Statistical Data Visualization & Flowmaps**  
- **Expert Consulting** tailored to customer infrastructure challenges  

Our solutions serve critical industries including Mobility, Retail & Services, Building & Campus Management, and Attractions & Destinations.

---

## Industries Served  

- **Mobility:** Public transport, traffic management, airport and ship terminals, stations, and asset management.  
- **Retail & Services:** Shopping centers, retail chains, and service-oriented environments.  
- **Building & Campus:** Smart building management and large campus optimization.  
- **Attraction & Destination:** Event management, tourism hubs, and large venues requiring crowd control and flow optimization.  

---

## Notable Clients and Impact  

ASE supports over 150 clients worldwide and manages more than 1,000,000 counts and flow assessments daily. Our client base includes renowned retailers, transport companies, and public authorities, who trust ASE to improve operational efficiency, safety, and visitor experience.  

Recent highlights:  
- Crowd management system at Lucerne Allmend/Messe station ensuring safe flow during UEFA Women’s EURO 2025 events.  
- Innovation in asset portfolio management and transport hubs featured in leading industry news.  
- Contributions to high-profile events such as Gurtenfestival.

---

## Company Culture & Team  

At ASE, we cultivate a culture of innovation, expert collaboration, and customer-centric problem solving. Our team in Zurich comprises multidisciplinary experts passionate about solving complex passenger flow and transport challenges through advanced technology and engineering.  

Employees enjoy working in a forward-thinking environment where continuous learning, cutting-edge research, and impactful real-world applications are encouraged.

---

## Careers at ASE  

Are you ready to join a pioneering team in AI, IoT, and transport solutions? ASE offers exciting career opportunities for professionals interested in technology, data science, engineering, and consulting within the mobility and infrastructure sectors.  

Join us to shape smarter, safer, and more efficient transport and public spaces for cities and communities worldwide.

---

## Contact ASE AG  

**Address:** Gartenhofstrasse 17, 8004 Zürich, Switzerland  
**Email:** info@ase.ch  
**Phone:** +41 (0)44 253 75 70  
**Website:** [ase.ch](https://ase.ch)

---

*Excellence through data-driven solutions — Intelligent decisions based on IoT, AI, and expert consulting.*  


## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [70]:
def stream_brochure(company_name, url):
    print(f"Creating brochure for {company_name} at {url} by calling gpt-4.1-mini")
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [71]:
stream_brochure("HuggingFace", "https://ase.ch")

Creating brochure for HuggingFace at https://ase.ch by calling gpt-4.1-mini
Selecting relevant links for https://ase.ch by calling claude-sonnet-4-5
Found 13 relevant links


# ASE AG – Intelligent AI & IoT Solutions for Mobility & Infrastructure

---

## About ASE

Founded in 1996 and headquartered in Zurich, Switzerland, ASE AG (Analysis Simulation Engineering) specializes in cutting-edge, data-driven solutions for public transport, mobility, retail, and built infrastructure. With a clear focus on maximizing the efficiency of existing systems, ASE helps clients leverage real-time data, AI technologies, and expert consulting to optimize space utilization, enhance safety, and improve customer experiences across diverse industries.

---

## What We Do

ASE delivers intelligent, AI- and IoT-powered solutions that empower clients to make smarter decisions. Our core expertise lies in:

- **Passenger Flow & Crowd Monitoring**  
- **People Counting & Vehicle Recognition**  
- **AI Detection & Simulation Tools**  
- **Expert Consulting & Data Visualization Platforms**  
- **Live Data & Statistical Analytics**  
- **Flow Mapping & Asset Portfolio Management**

Our approach centers around unlocking the full potential of transport hubs, retail centers, public spaces, and event venues—tackling complex challenges without costly infrastructure expansion.

---

## Industries We Serve

- **Mobility:** Stations, public transport, traffic management, airports, ships  
- **Retail & Services:** Shopping centers, retail chains  
- **Building & Campus:** Office buildings, university campuses  
- **Attraction & Destination:** Events, tourism destinations, festivals

---

## Our Customers & Impact

We proudly serve over 150 clients, including major retailers, transport companies, and public authorities, handling upwards of 1,000,000 counts per day. Our systems are trusted for vital projects such as crowd management during UEFA Women’s EURO 2025 and asset portfolio optimization in transport hubs.  

---

## The ASE Team & Culture

ASE is a close-knit team of 30 dedicated experts passionate about transport and flow solutions. We blend technology, research, and engineering know-how to provide innovative, practical answers that maximize client outcomes while safeguarding user safety and satisfaction.

Our culture values:

- Technical excellence in AI and IoT  
- Collaborative problem-solving with clients  
- Continuous research and development  
- Commitment to sustainable, cost-effective solutions  

---

## Careers at ASE

Join ASE to work at the intersection of AI, IoT, and transport innovation. We seek talented professionals who want to make a tangible impact on public infrastructure and urban mobility. Being part of ASE means engaging in meaningful projects, working within a motivated team environment, and contributing to the future of intelligent transport solutions.

For current job openings and application details, visit our website or contact us directly.

---

## Contact

**ASE AG (Analysis Simulation Engineering) AG**  
Gartenhofstrasse 17 | 8004 Zürich | Switzerland  
Email: info@ase.ch  
Phone: +41 (0)44 253 75 70  
Website: [www.ase.ch](https://www.ase.ch)  

---

Empowering Mobility & Infrastructure Through Intelligent, Data-Driven Solutions.  
Experience the ASE advantage—where technology meets expertise for smarter, safer, and more efficient public spaces.

In [22]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links


# Welcome to Hugging Face – The AI Community Building the Future!

---

## Who Are We?  
Imagine a place where machine learning wizards, data sorcerers, and AI alchemists gather to share their spells — uh, models — datasets, and apps. That’s Hugging Face! We’re *the* platform where the AI community collaborates, creates, and sometimes even has a little fun while building the future.

Our motto? **"Keep it open. Keep it ethical. Keep it hugging."** 💛

---

## What’s Cooking in the AI Kitchen?

- **1 Million+ Models** — From image generators to language wizards, our treasure trove of open-source ML models grows faster than you can say "neural network."  
- **250,000+ Datasets** — Feeding AI brains with everything from chat prompts to persona profiles. Hungry for data? Dig in!  
- **400,000+ Applications & Spaces** — Launch apps, share your ML portfolio, or just show off cool demos that make your friends say, “Whoa, AI can do that?”  
- **Multimodal Madness** — Text, image, video, audio, even 3D...if AI had a Swiss Army knife, we’d be it.  

---

## Customers & Community  
Whether you’re a student trying to get your AI feet wet, a startup looking to scale your genius, or an enterprise aiming to deploy heavy-duty models in the real world, Hugging Face has your back.

With the fastest growing community of *machine learning enthusiasts* and the support of some seriously big names and organizations, here’s a place where:

- **Freelancers** can build a portfolio and get noticed.  
- **Researchers** can push boundaries openly and ethically.  
- **Businesses** can accelerate AI adoption with our paid Compute and Enterprise suites.  

Join 1.29k+ Spaces and thousands more running models that power everything from video generation to AI-powered image editing.

---

## Culture & Career – Geek Out with Us!  
We believe collaboration beats isolation every day. Our culture?

- Open source at heart ❤️  
- Ethical AI advocates  
- Casual tea-drinkers and serious problem solvers  
- Always learning, always sharing, always growing  

Want to build machine learning tools that millions will use? Hugging Face is where your skills meet endless possibilities. From ML engineers to community managers, our doors are wide open (virtual hugs included).

---

## Speed Up Your AI Journey  
No need to code in the dark alone or fight for GPU time — deploy models and apps with a few clicks on optimized inference endpoints, starting at just $0.60/hour for GPU!

Whether you want to host that killer new model or just tweak an existing one, we give you the tools and community support to **move faster, build smarter, and hug tighter**.

---

## Quick Hugging Face Facts  
- **Founded:** Around the corner from the future  
- **Colors:** Bright yellow (#FFD21E), orange (#FF9D00), and sleek gray (#6B7280) — because AI should be as vibrant as its ideas!  
- **Mascot:** Friendly face with a warm smile (because AIs could learn a thing or two about friendliness here)  

---

## Ready to Join the AI Hug Circle?  

Sign up, share your work, explore millions of models and datasets, and get your AI career (or project!) hugging new heights.

[Explore AI Apps](#) | [Browse 1M+ Models](#) | [Sign Up & Join The Fun](#)

---

*Hugging Face — where the future of AI isn’t just created; it’s hugged into existence.* 🤗✨

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>